# TextRank Algorithm for Extractive Text Summarization

In [4]:
import spacy
import pytextrank

nlp = spacy.load("en_core_web_lg")
nlp.add_pipe("textrank")

In [5]:
example_text = """Deep learning (also known as deep structured learning) is part of a 
broader family of machine learning methods based on artificial neural networks with 
representation learning. Learning can be supervised, semi-supervised or unsupervised. 
Deep-learning architectures such as deep neural networks, deep belief networks, deep reinforcement learning, 
recurrent neural networks and convolutional neural networks have been applied to
fields including computer vision, speech recognition, natural language processing, 
machine translation, bioinformatics, drug design, medical image analysis, material
inspection and board game programs, where they have produced results comparable to 
and in some cases surpassing human expert performance. Artificial neural networks
(ANNs) were inspired by information processing and distributed communication nodes
in biological systems. ANNs have various differences from biological brains. Specifically, 
neural networks tend to be static and symbolic, while the biological brain of most living organisms
is dynamic (plastic) and analogue. The adjective "deep" in deep learning refers to the use of multiple
layers in the network. Early work showed that a linear perceptron cannot be a universal classifier, 
but that a network with a nonpolynomial activation function with one hidden layer of unbounded width can.
Deep learning is a modern variation which is concerned with an unbounded number of layers of bounded size, 
which permits practical application and optimized implementation, while retaining theoretical universality 
under mild conditions. In deep learning the layers are also permitted to be heterogeneous and to deviate widely 
from biologically informed connectionist models, for the sake of efficiency, trainability and understandability, 
whence the structured part."""
print('Original Document Size:',len(example_text))

Original Document Size: 1820


In [6]:
doc  = nlp(example_text)

for sent in doc._.textrank.summary(limit_phrases=2, limit_sentences=2):
    print(sent)
    print('Summary Length:', len(sent))

Deep-learning architectures such as deep neural networks, deep belief networks, deep reinforcement learning, 
recurrent neural networks and convolutional neural networks have been applied to
fields including computer vision, speech recognition, natural language processing, 
machine translation, bioinformatics, drug design, medical image analysis, material
inspection and board game programs, where they have produced results comparable to 
and in some cases surpassing human expert performance.
Summary Length: 81
Specifically, 
neural networks tend to be static and symbolic, while the biological brain of most living organisms
is dynamic (plastic) and analogue.
Summary Length: 29


### Summarizes a Wikipedia article based on (a) ratio and (b) word count.

In [12]:
import spacy
import pytextrank
import wikipedia
import en_core_web_sm

nlp = spacy.load('en_core_web_lg')
nlp.add_pipe('textrank')

In [13]:
wikisearch = wikipedia.page("Amitabh Bachchan")
wikicontent = wikisearch.content

doc = nlp(wikicontent)

In [14]:
f = open('wikicontent.txt', "w")
f.write(wikicontent)
f.close()

In [15]:
print("Summary")

for sent in doc._.textrank.summary(limit_sentences=3):
    print(sent)
    

Summary
A review in Daily News and Analysis (DNA) summarised Bachchan's performance as "The heart and soul of Piku clearly belong to Amitabh Bachchan who is in his elements.
Writing for Hindustan Times, noted film critic and author Anupama Chopra said of Bachchan's performance, "A special salute to Amitabh Bachchan, who imbues his character with a tragic majesty.
In the early 80s, Bachchan authorised the use of his likeness for the comic book character Supremo in a series titled The Adventures of Amitabh Bachchan.


In [16]:
print('\n Top Keywords')

for phrase in doc._.phrases[:10]:
    print(phrase.text)


 Top Keywords
Amitabh Bachchan
Bachchan
Amitabh Bachchan Road
Amitabh Bachchan Falls
Mr. Bachchan
Ajitabh Bachchan
Jaya Bachchan
Bachchan towers
Teji Bachchan
Amitabh Bachchan Corporation Ltd


# Abstractive Summarization

In [17]:
from transformers import pipeline
from transformers import PegasusForConditionalGeneration, PegasusTokenizer


In [18]:

model_name = "google/pegasus-xsum"

pegasus_tokenizer = PegasusTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

In [19]:
example_text = """
Deep learning (also known as deep structured learning) is part of a broader family of machine learning
methods based on artificial neural networks with representation learning. 
Learning can be supervised, semi-supervised or unsupervised. Deep-learning architectures such as 
deep neural networks, deep belief networks, deep reinforcement learning, 
recurrent neural networks and convolutional neural networks have been applied to 
fields including computer vision, speech recognition, natural language processing,
machine translation, bioinformatics, drug design, medical image analysis, 
material inspection and board game programs, where they have produced results 
comparable to and in some cases surpassing human expert performance. 
Artificial neural networks (ANNs) were inspired by information processing and 
distributed communication nodes in biological systems. ANNs have various differences 
from biological brains. Specifically, neural networks tend to be static and symbolic,
while the biological brain of most living organisms is dynamic (plastic) and analogue.
The adjective "deep" in deep learning refers to the use of multiple layers in the network.
Early work showed that a linear perceptron cannot be a universal classifier, 
but that a network with a nonpolynomial activation function with one hidden layer of 
unbounded width can. Deep learning is a modern variation which is concerned with an 
unbounded number of layers of bounded size, which permits practical application and 
optimized implementation, while retaining theoretical universality under mild conditions. 
In deep learning the layers are also permitted to be heterogeneous and to deviate widely 
from biologically informed connectionist models, for the sake of efficiency, trainability 
and understandability, whence the structured part."""

print('Original Document Size:',len(example_text))

Original Document Size: 1825


In [20]:
pegasus_model = PegasusForConditionalGeneration.from_pretrained(model_name)

tokens = pegasus_tokenizer(example_text, truncation=True, padding="longest", return_tensors="pt")


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
encoded_summary = pegasus_model.generate(**tokens)

In [ ]:
decoded_summary = pegasus_tokenizer.decode(encoded_summary[0], skip_special_tokens=True)
print("Decoded Summary : ",decoded_summary)

In [ ]:
summarizer = pipeline(
    "summarization",
    model=model_name,
    tokenizer=pegasus_tokenizer,
    framework="pt"
)

In [ ]:
summary = summarizer(example_text, min_length=30, max_length=150)
summary[0]['summary_text']

### BART for summarization

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

model_name = 'facebook/bart-large-cnn'

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

In [ ]:
text = " "

In [ ]:
inputs = tokenizer(text, return_tensors='pt', max_length=1024, truncation=True)

summary_ids = model.generate(
    inputs['input_ids'],
    max_length=60,
    min_length=20,
    length_penalty=2.0,
    num_beams=4
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)

### BERT for Summarization

#### when Factual Accuracy matters!

In [ ]:
#High Level Wrapper
from summarizer import Summarizer
model = Summarizer()
text = " "
summary = model(text, num_sentences=2)
print(summary)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as nlp

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model - AutoModel.from_pretrained(model_name)

In [ ]:
text = " "

In [ ]:
sentences = text.strip().split("\n")

embeddings = []

for sentence in sentences:
    inputs = tokenizer(sentence, return_tensor='pt', truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1).squeeze()
    embeddings.append(embedding)

embeddings = torch.stack(embeddings)

scores = embeddings @ embeddings.mean(dim=0)

top_idx = torch.argsort(scores, descending=True)[:2]

summary = [sentences[i] for i in top_idx]

print("Summary:")
for s in summary:
    print(s)